# Geographic N-Gram Dialect Identification — Colab Experiments

This notebook runs all three training conditions from the paper and produces the main results table.

| Condition | Training data | Time |
|-----------|--------------|------|
| `oracle` | NADI gold labels (21K tweets, human-annotated) | ~1 min |
| `geo_twitter` | 440K tweets by user geo-declaration (HuggingFace) | ~10 min |
| `cc_tld` | OSCAR web text filtered by country TLD | ~2–4 hours |

**No GPU needed.** All computation is character n-gram (pure CPU).

---

## Before you start — put these in Google Drive

Upload your `tinylid/` folder to `My Drive/tinylid/`. The directory should look like:

```
My Drive/
  tinylid/
    train_geo.py
    eval_dialect.py
    hierarchy.py
    ngram_lm.py
    ...
    data/
    baselines/
    datasets/
      nadi2020/         ← upload the NADI data here
        NADI2020_shared_task/
          NADI_release/
            train_labeled.tsv
            dev_labeled.tsv
    models/             ← trained models will be saved here
```

Models are saved to Drive so they persist when the session ends.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, sys

DRIVE_TINYLID = '/content/drive/MyDrive/tinylid'
WORK_DIR      = '/content/tinylid'

# Copy from Drive to /content/ so imports work cleanly
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(DRIVE_TINYLID, WORK_DIR)

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print('Working directory:', os.getcwd())
print('Files:', [f for f in os.listdir() if not f.startswith('__')])

In [ ]:
# Install dependencies
# numpy<2.0 is required — fasttext-wheel breaks with numpy 2.x
!pip install -q 'numpy<2.0' fasttext-wheel huggingface-hub datasets tqdm
print('Done.')

In [ ]:
# Quick sanity check — all core modules should import cleanly
import ngram_lm, hierarchy, save_load
from data import nadi, geo_langs, arabic_twitter_hf
print('Core imports OK')
print('Supported languages:', list(geo_langs.LANGUAGES.keys())[:6])

## 2. Check data availability

In [ ]:
from pathlib import Path

NADI_DIR    = Path('datasets/nadi2020/NADI2020_shared_task/NADI_release')
MODELS_DIR  = Path('models')
MODELS_DIR.mkdir(exist_ok=True)

nadi_ok = (NADI_DIR / 'train_labeled.tsv').exists()
print(f'NADI data found:  {nadi_ok}  ({NADI_DIR})')

if nadi_ok:
    from data.nadi import load_split
    train_ex = load_split(NADI_DIR, 'train')
    dev_ex   = load_split(NADI_DIR, 'dev')
    from collections import Counter
    cc = Counter(e.country for e in train_ex if e.country)
    print(f'  Train: {len(train_ex):,} tweets, {len(cc)} countries')
    print(f'  Dev:   {len(dev_ex):,} tweets')
    print(f'  Countries: {sorted(cc.keys())}')
else:
    print('  NADI data missing — oracle experiment will be skipped.')
    print('  Upload it to Drive/tinylid/datasets/nadi2020/ and re-run this cell.')

## 3. Experiment A — Oracle (NADI gold labels)

Upper bound: trained on human-annotated dialect labels. ~1 minute.

In [ ]:
if not nadi_ok:
    print('Skipping oracle — NADI data not found.')
else:
    !python train_geo.py \
        --language ar \
        --source nadi \
        --data-dir {NADI_DIR} \
        --order 4 \
        --out models/ar_oracle.pkl
    print('\nOracle model saved → models/ar_oracle.pkl')

## 4. Experiment B — Geo-Twitter (weak supervision, HuggingFace)

Trains on 440K tweets where the country label comes from the user's Twitter profile
description — **no human dialect annotation**. Downloads automatically. ~10 minutes.

Dataset: `Abdelrahman-Rezk/Arabic_Dialect_Identification`  
18 countries: EG, SA, MA, DZ, TN, LY, SD, IQ, SY, LB, JO, PS, QA, KW, AE, YE, OM, BH

In [ ]:
# Step 1: cache the HuggingFace dataset to disk (so we can re-use without re-downloading)
TWITTER_CACHE = Path('datasets/arabic_twitter_hf')

if TWITTER_CACHE.exists() and any(TWITTER_CACHE.glob('*.txt')):
    print(f'Cache already exists at {TWITTER_CACHE} — skipping download.')
    cached = list(TWITTER_CACHE.glob('*.txt'))
    print(f'  {len(cached)} country files found: {sorted(p.stem for p in cached)}')
else:
    print('Downloading from HuggingFace (first time only, ~5 min)...')
    !python -m data.arabic_twitter_hf --out-dir datasets/arabic_twitter_hf

    # Copy cache back to Drive so we never re-download
    drive_cache = Path(DRIVE_TINYLID) / 'datasets/arabic_twitter_hf'
    if not drive_cache.exists():
        shutil.copytree(TWITTER_CACHE, drive_cache)
        print(f'Saved cache to Drive → {drive_cache}')

In [ ]:
# Step 2: train n-gram models on the cached Twitter data
!python train_geo.py \
    --language ar \
    --source aratweet_hf \
    --data-dir datasets/arabic_twitter_hf \
    --order 4 \
    --out models/ar_geo_twitter.pkl

print('\nGeo-Twitter model saved → models/ar_geo_twitter.pkl')

## 5. Experiment C — CC-TLD (OSCAR web corpus)

Trains on Arabic web text filtered by country domain (`.eg`, `.sa`, `.ma`, ...).
**No labels at all** — purely URL-based signal.

⚠️ This streams through OSCAR-2201 Arabic (~2.9M documents). Takes **2–4 hours**.
- Free Colab: run overnight, or skip for now and do it on CINECA CPU nodes
- Colab Pro: stable long session, recommended

Skip this cell if you want fast results — the geo_twitter model is the main contribution.

In [ ]:
# Set RUN_CCTLD = True to stream OSCAR (takes hours)
RUN_CCTLD = False

CCTLD_CACHE = Path('datasets/cc_ar')

if not RUN_CCTLD:
    print('CC-TLD skipped (set RUN_CCTLD = True to enable).')
    print('Run this on CINECA CPU nodes or Colab Pro with a stable session.')
elif CCTLD_CACHE.exists() and any(CCTLD_CACHE.glob('*.txt')):
    print(f'CC-TLD cache found at {CCTLD_CACHE} — using existing files.')
    !python train_geo.py \
        --language ar \
        --source cc_tld \
        --data-dir datasets/cc_ar \
        --order 4 \
        --out models/ar_cctld.pkl
else:
    print('Streaming OSCAR Arabic from HuggingFace (this will take 2-4 hours)...')
    # Stream and write to disk first so training doesn't need to re-stream
    !python -m data.cc_tld \
        --language ar \
        --out-dir datasets/cc_ar \
        --max-per-country 50000

    # Save cache to Drive
    drive_cc = Path(DRIVE_TINYLID) / 'datasets/cc_ar'
    if not drive_cc.exists():
        shutil.copytree(CCTLD_CACHE, drive_cc)
        print(f'Saved to Drive → {drive_cc}')

    !python train_geo.py \
        --language ar \
        --source cc_tld \
        --data-dir datasets/cc_ar \
        --order 4 \
        --out models/ar_cctld.pkl
    print('CC-TLD model saved → models/ar_cctld.pkl')

## 6. Download fastText and GlotLID baselines

In [ ]:
# Auto-downloads on first use if not already present
from pathlib import Path

FT_PATH   = Path('models/model.bin')
GLOT_PATH = Path('models/glotlid/model.bin')

if not FT_PATH.exists():
    print('Downloading fastText lid.218...')
    from baselines.fasttext_lid import FasttextLID
    _ = FasttextLID()   # triggers auto-download
    print('  Done.')
else:
    print(f'fastText already at {FT_PATH}')

if not GLOT_PATH.exists():
    print('Downloading GlotLID v3...')
    from baselines.fasttext_lid import GlotLID
    _ = GlotLID()       # triggers auto-download
    print('  Done.')
else:
    print(f'GlotLID already at {GLOT_PATH}')

## 7. Evaluate — Table 1 of the paper

Evaluates all trained models + baselines on NADI 2020 dev (4,957 tweets, 21 countries).

In [ ]:
if not nadi_ok:
    print('Cannot evaluate — NADI dev data not found. Upload it to Drive first.')
else:
    # Build --models argument from whichever .pkl files exist
    model_specs = []
    for path, name in [
        ('models/ar_oracle.pkl',      'oracle'),
        ('models/ar_geo_twitter.pkl', 'geo_twitter'),
        ('models/ar_cctld.pkl',       'cc_tld'),
    ]:
        if Path(path).exists():
            model_specs.append(f'{path}:{name}')
        else:
            print(f'  [skip] {name} model not found at {path}')

    models_arg = ' '.join(model_specs)
    print(f'Evaluating: {[s.split(":")[1] for s in model_specs]}')
    print()

    !python eval_dialect.py \
        --language ar \
        --eval-source nadi \
        --eval-dir {NADI_DIR} \
        --split dev \
        --models {models_arg} \
        --per-variety

## 8. Per-variety breakdown (heatmap)

Shows which countries each model can and cannot distinguish.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from eval_dialect import _load_eval_data, compute_metrics, _predict_hier, _predict_baseline
from save_load import load as load_bundle

if not nadi_ok:
    print('Skipping heatmap — NADI data not found.')
else:
    texts, gold = _load_eval_data('nadi', NADI_DIR, 'ar', 'dev')
    varieties = sorted(set(gold))

    # Collect per-variety F1 for each available model
    all_results = {}

    # Baselines
    try:
        from baselines.fasttext_lid import FasttextLID, GlotLID
        ft   = FasttextLID()
        glot = GlotLID()
        all_results['fastText'] = compute_metrics(gold, ft.predict_batch(texts),   varieties)
        all_results['GlotLID']  = compute_metrics(gold, glot.predict_batch(texts), varieties)
    except Exception as e:
        print(f'Baseline error: {e}')

    # Trained models
    for path, name in [
        ('models/ar_oracle.pkl',      'oracle'),
        ('models/ar_geo_twitter.pkl', 'geo_twitter'),
        ('models/ar_cctld.pkl',       'cc_tld'),
    ]:
        if Path(path).exists():
            hier = load_bundle(Path(path)).hierarchy
            preds = _predict_hier(hier, texts)
            all_results[name] = compute_metrics(gold, preds, varieties)

    if not all_results:
        print('No models found to plot.')
    else:
        model_names = list(all_results.keys())
        matrix = np.array([[all_results[m]['per_variety'].get(v, 0.0)
                             for v in varieties]
                            for m in model_names])

        fig, ax = plt.subplots(figsize=(14, 0.5 * len(model_names) + 2))
        im = ax.imshow(matrix, aspect='auto', vmin=0, vmax=1,
                       cmap='RdYlGn', interpolation='nearest')

        ax.set_xticks(range(len(varieties)))
        ax.set_xticklabels(varieties, rotation=45, ha='right', fontsize=9)
        ax.set_yticks(range(len(model_names)))
        ax.set_yticklabels(model_names, fontsize=10)

        for i, m in enumerate(model_names):
            for j, v in enumerate(varieties):
                val = matrix[i, j]
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=7, color='black' if val > 0.4 else 'gray')

        plt.colorbar(im, ax=ax, label='F1 score')
        ax.set_title('Per-country F1  |  Arabic Dialect Identification (NADI 2020 dev)')
        plt.tight_layout()
        plt.savefig('results_heatmap.pdf', bbox_inches='tight')
        plt.show()
        print('Saved → results_heatmap.pdf')

## 9. Summary table (clean printout)

In [ ]:
if 'all_results' not in dir() or not all_results:
    print('Run cell 8 first to populate all_results.')
else:
    print(f'\n{"="*65}')
    print(f'Arabic Dialect Identification  |  NADI 2020 dev  |  {len(varieties)} countries')
    print(f'{"="*65}')
    print(f'{"Model":<20} {"Acc":>7} {"MacroF1":>9} {"CovF1":>8} {"Coverage":>10}')
    print('-' * 58)
    for name, r in all_results.items():
        print(f'{name:<20} '
              f'{r["accuracy"]:>7.3f} '
              f'{r["macro_f1"]:>9.3f} '
              f'{r["covered_f1"]:>8.3f} '
              f'{r["n_covered"]:>4}/{r["n_varieties"]}')
    print()
    print('Acc      = % correct')
    print('MacroF1  = mean F1 across ALL varieties (0 for uncovered)')
    print('CovF1    = mean F1 for varieties the model actually predicts')
    print('Coverage = how many of the 21 countries this model can distinguish')

## 10. Save models back to Google Drive

In [ ]:
import shutil
from pathlib import Path

drive_models = Path(DRIVE_TINYLID) / 'models'
drive_models.mkdir(exist_ok=True)

saved = []
for pkl in Path('models').glob('*.pkl'):
    dst = drive_models / pkl.name
    shutil.copy2(pkl, dst)
    saved.append(pkl.name)

if saved:
    print(f'Saved to Drive ({len(saved)} models):')
    for f in saved:
        print(f'  {f}')
else:
    print('No .pkl models found to save.')

# Also save the heatmap if it was generated
if Path('results_heatmap.pdf').exists():
    shutil.copy2('results_heatmap.pdf', Path(DRIVE_TINYLID) / 'results_heatmap.pdf')
    print('Saved results_heatmap.pdf to Drive.')

---

## Bonus — Spanish / Portuguese / English

The same pipeline works for the other languages using DSL-TL data.
Requires: download DSL-TL from GitHub (`git clone https://github.com/LanguageTechnologyLab/DSL-TL`)
and place in `datasets/DSL-TL/`.

In [ ]:
DSL_DIR = Path('datasets/DSL-TL')

if not DSL_DIR.exists():
    print('Cloning DSL-TL from GitHub...')
    !git clone https://github.com/LanguageTechnologyLab/DSL-TL datasets/DSL-TL
else:
    print('DSL-TL already present.')

for lang in ['pt', 'es', 'en']:
    out = Path(f'models/{lang}_oracle.pkl')
    if out.exists():
        print(f'{lang}: model already trained at {out}')
        continue
    print(f'\nTraining {lang} oracle...')
    !python train_geo.py \
        --language {lang} \
        --source dsltl \
        --data-dir datasets/DSL-TL \
        --order 4 \
        --out models/{lang}_oracle.pkl

In [ ]:
# Evaluate PT, ES, EN
if DSL_DIR.exists():
    for lang in ['pt', 'es', 'en']:
        model_path = f'models/{lang}_oracle.pkl'
        if not Path(model_path).exists():
            continue
        print(f'\n--- {lang.upper()} ---')
        !python eval_dialect.py \
            --language {lang} \
            --eval-source dsltl \
            --eval-dir datasets/DSL-TL \
            --split dev \
            --models {model_path}:oracle \
            --no-fasttext --no-glotlid